# Basic Usage

This notebook contains some basic examples with python code for accessing the TROVE API

It assumes you have the requests and pandas packages installed and the following two environment variables defined

1. `TROVE_USER = <your_username>`
2. `TROVE_PASS = <your_password>`

## Imports & Useful Variables

In [1]:
import os
import requests
import pandas as pd

trove_url = "https://datatrove-test.as.arizona.edu/api/"
username = os.environ.get("TROVE_USER")
password = os.environ.get("TROVE_PASS")
nle_name = "GW190814" # a nonlocalized event (e.g., GW event) name

## Getting Scores

The primary thing users want to do is download the scores via the API to help them prioritize follow-up. To get all scores of candidates associated with a transient we can use the "score" endpoint: 

In [2]:
response = requests.get(
    f"{trove_url}/score/{nle_name}",
    auth=(username, password),
    headers={"accept": "*/*"}
)

scores = response.json()

scores[0]

{'id': 3845,
 'target': {'id': 1368128219,
  'name': 'RAPTORS-blu',
  'ra': 12.22920833,
  'dec': -25.06973472,
  'tns_redshift': None,
  'tns_classification': None},
 'nonlocalizedevent': {'id': 21085,
  'event_id': 'GW190814',
  'event_type': 'GW'},
 'score': {'KN': 0.780326475350289}}

Which is the highest scoring candidate. The dictionary has a few important keys

1. The candidate "id". This will be unique for an NLE
2. `target`: A dictionary of target information including the TNS (or closest) name, ra, dec, tns_redshift, tns_classification
3. `nonlocalizedevent`: A dictionary with information about the NLE
4. `score`: A dictionary with keys for different scoring algorithms. For NS bearing GW events this will always have the key "KN". For SSM events it may also have keys for "KN-in-SN" and "super-KN" vetting algorithms.

But, we can also get the lowest scoring candidate:

In [3]:
scores[-1]

{'id': 3089,
 'target': {'id': 41709244,
  'name': 'AT2019nuy',
  'ra': 12.7099877991,
  'dec': -25.4915413351,
  'tns_redshift': nan,
  'tns_classification': ''},
 'nonlocalizedevent': {'id': 21085,
  'event_id': 'GW190814',
  'event_type': 'GW'},
 'score': {'KN': 0.0}}

Or, the top ten scores:

In [4]:
scores[:10]

[{'id': 3845,
  'target': {'id': 1368128219,
   'name': 'RAPTORS-blu',
   'ra': 12.22920833,
   'dec': -25.06973472,
   'tns_redshift': None,
   'tns_classification': None},
  'nonlocalizedevent': {'id': 21085,
   'event_id': 'GW190814',
   'event_type': 'GW'},
  'score': {'KN': 0.780326475350289}},
 {'id': 3846,
  'target': {'id': 1368128220,
   'name': 'RAPTORS-red',
   'ra': 12.22920833,
   'dec': -25.06973472,
   'tns_redshift': None,
   'tns_classification': None},
  'nonlocalizedevent': {'id': 21085,
   'event_id': 'GW190814',
   'event_type': 'GW'},
  'score': {'KN': 0.780326475350289}},
 {'id': 3853,
  'target': {'id': 1368500633,
   'name': 'mock',
   'ra': 12.1,
   'dec': -25.3,
   'tns_redshift': None,
   'tns_classification': None},
  'nonlocalizedevent': {'id': 21085,
   'event_id': 'GW190814',
   'event_type': 'GW'},
  'score': {'KN': 0.562660740049493}},
 {'id': 3064,
  'target': {'id': 41708423,
   'name': 'AT2019pck',
   'ra': 11.851431586385,
   'dec': -25.22605663008

If we instead only want the score of a specific transient we can use some query parameters in the URL.

In [5]:
response = requests.get(
    f"{trove_url}/score/{nle_name}?candidate_names=AT2019pck",
    auth=(username, password),
    headers={"accept": "*/*"}
)

at2019pck_score = response.json()
at2019pck_score

[{'id': 3064,
  'target': {'id': 41708423,
   'name': 'AT2019pck',
   'ra': 11.851431586385,
   'dec': -25.226056630082,
   'tns_redshift': nan,
   'tns_classification': ''},
  'nonlocalizedevent': {'id': 21085,
   'event_id': 'GW190814',
   'event_type': 'GW'},
  'score': {'KN': 0.5441244282667317}}]

Or, say we are interested in a few different targets

In [6]:
response = requests.get(
    f"{trove_url}/score/{nle_name}?candidate_names=AT2019pck,AT2019odc",
    auth=(username, password),
    headers={"accept": "*/*"}
)

scores = response.json()
scores

[{'id': 3064,
  'target': {'id': 41708423,
   'name': 'AT2019pck',
   'ra': 11.851431586385,
   'dec': -25.226056630082,
   'tns_redshift': nan,
   'tns_classification': ''},
  'nonlocalizedevent': {'id': 21085,
   'event_id': 'GW190814',
   'event_type': 'GW'},
  'score': {'KN': 0.5441244282667317}},
 {'id': 3036,
  'target': {'id': 41709056,
   'name': 'AT2019odc',
   'ra': 11.507039,
   'dec': -25.45915,
   'tns_redshift': nan,
   'tns_classification': ''},
  'nonlocalizedevent': {'id': 21085,
   'event_id': 'GW190814',
   'event_type': 'GW'},
  'score': {'KN': 0.258832951381733}}]

## Getting Scores with a Cone Search

Say you are trying to plan an observing follow-up campaign and you want to prioritize the top 10 targets that are visible to your telescope. To do this, you can get a ranked list of candidates with a cone search:

In [7]:
ra_ctr = 13.163209
dec_ctr = -25.004159
radius = 3600 # 5 arcmin

response = requests.get(
    f"{trove_url}/score/{nle_name}/cone_search?ra={ra_ctr}&dec={dec_ctr}&radius={radius}",
    auth=(username, password),
    headers={"accept": "*/*"}
)

scores = response.json()
len(scores)

23

## Uploading New Targets & Photometry

Finally, another common use case is reporting new targets and/or photometry that you got! For this, we can use the `target` endpoint.

In [8]:
upload_url = f"{trove_url}/target/upload"

target_data = [
    {
        "name": "test",
        "ra": 0,
        "dec": 0
    }
]

response = requests.post(
    upload_url,
    json=target_data,
    auth=(username, password),
    headers={"accept": "*/*"}
)

print(response.status_code)

200


Or, if you want to also provide photometry

In [9]:
target_data = [
  {
    "name": "test",
    "ra": 0,
    "dec": 0,
    "photometry": [
      {
        "jd": 2461134.3933751667,
        "telescope": "TEST",
        "filter": "r",
        "magnitude": 20,
        "error": 0.1
      },
      {
        "jd": 2461133.3933751667,
        "telescope": "TEST",
        "filter": "r",
        "limit": 23.1
      }
    ]
  }
]

response = requests.post(
    upload_url,
    json=target_data,
    auth=(username, password),
    headers={"accept": "*/*"}
)

print(response.status_code)

200
